In [4]:
from pathlib import Path
import zipfile

ruta = Path("../Datos/originales/")

print("¿Existe la carpeta?:", ruta.exists())

archivos = list(ruta.glob("*"))

for archivo in archivos:
    print(archivo.name)

    if archivo.suffix.lower() == ".zip":
        with zipfile.ZipFile(archivo) as zip:
            for nombre in zip.namelist():
                print("  ", nombre)

¿Existe la carpeta?: True
.gitkeep
denue
mercados
tianguis


In [6]:
archivo = ruta / "tianguis_cdmx_sedeco.shp"

with open(archivo, "rb") as f:
    print("Primeros bytes:", f.read(16))

print("Tamaño:", archivo.stat().st_size, "bytes")

Primeros bytes: b'PK\x03\x04\x14\x00\x00\x00\x00\x00\x8b\x84\x87RP<'
Tamaño: 115813 bytes


In [8]:
from pathlib import Path

ruta = Path("../Datos/originales/tianguis")

for archivo in ruta.iterdir():
    if archivo.is_file():
        print(archivo.name, "|", archivo.stat().st_size, "bytes")

datos.csv | 210564 bytes
diccionario.csv | 1006 bytes
tianguis_cdmx_sedeco.shp | 115813 bytes


In [9]:
import pandas as pd

ruta = "../Datos/originales/tianguis/datos.csv"

df_tianguis = pd.read_csv(ruta, encoding="utf-8-sig")

print("Filas:", len(df_tianguis))
print("Columnas:", len(df_tianguis.columns))

print("\nNombres de las columnas:")
print(df_tianguis.columns.tolist())

display(df_tianguis.head())

Filas: 1367
Columnas: 8

Nombres de las columnas:
['alcaldia', 'colonia', 'ubicacion', 'giro', 'dia', 'horario', 'latitud', 'longitud']


,alcaldia,colonia,ubicacion,giro,dia,horario,latitud,longitud
0,ALVARO OBREGON,PILOTO (ADOLFO LOPEZ MATEOS) (AMPL),ADOLFO DE LA HUERTA ENTRE RUIZ CORTINEZ Y BENI...,NaN,LUNES,8:00 A 17:00 HORAS,19.375612,-99.221648
1,ALVARO OBREGON,GOLONDRINAS 1RA SECCION,AV. CHICAGO ENTRE URANO Y JUPITER,NaN,LUNES,8:00 A 17:00 HORAS,19.381123,-99.209121
2,ALVARO OBREGON,JOSE MARIA PINO SUAREZ,PONIENTE 85 Y AV. OBSERVATORIO,NaN,LUNES,8:00 A 17:00 HORAS,19.398888,-99.197936
3,ALVARO OBREGON,TLACUITLAPA,CALLE LAS TORRES Y ANTONIO MACHADO,NaN,LUNES,8:00 A 17:00 HORAS,19.355768,-99.240129
4,ALVARO OBREGON,MARIA G DE GARCIA RUIZ,AV. MEXICANOS ENT. BRASILEÃ‘OS Y URUGUAYOS,NaN,LUNES,8:00 A 17:00 HORAS,19.385972,-99.212968


In [10]:
# Valores faltantes por columna
print("VALORES FALTANTES")
print(df_tianguis.isna().sum())

# Registros duplicados
print("\nDUPLICADOS")
print(df_tianguis.duplicated().sum())

# Días de operación
print("\nDÍAS DE OPERACIÓN")
print(df_tianguis["dia"].value_counts(dropna=False))

# Horarios registrados
print("\nHORARIOS")
print(df_tianguis["horario"].value_counts(dropna=False))

# Alcaldías
print("\nALCALDÍAS")
print(df_tianguis["alcaldia"].value_counts(dropna=False))

VALORES FALTANTES
alcaldia        0
colonia         0
ubicacion       0
giro         1053
dia             0
horario         0
latitud         0
longitud        0
dtype: int64

DUPLICADOS
33

DÍAS DE OPERACIÓN
dia
DOMINGO                                                        259
MARTES                                                         195
SÁBADO                                                         192
JUEVES                                                         190
LUNES                                                          189
MIÉRCOLES                                                      172
VIERNES                                                        159
SÁBADO, DOMINGO                                                  3
LUNES, MARTES, MIÉRCOLES, JUEVES, VIERNES, SÁBADO, DOMINGO       2
LUNES, VIERNES                                                   1
LUNES, MIÉRCOLES, SÁBADO, DOMINGO                                1
MARTES, SÁBADO                                    

In [11]:
ruta_csv = "../Datos/originales/tianguis/datos.csv"

with open(ruta_csv, "rb") as f:
    contenido = f.read()

for codificacion in ["utf-8-sig", "cp1252"]:
    try:
        texto = contenido.decode(codificacion)
        print(codificacion, ":", texto[:200])
    except UnicodeDecodeError:
        print(codificacion, ": no compatible")

utf-8-sig : alcaldia,colonia,ubicacion,giro,dia,horario,latitud,longitud
ALVARO OBREGON,PILOTO (ADOLFO LOPEZ MATEOS) (AMPL),ADOLFO DE LA HUERTA ENTRE RUIZ CORTINEZ Y BENITO JUAREZ,NA,LUNES,8:00 A 17:00 HORAS,19.3
cp1252 : no compatible


In [12]:
import unicodedata

df_tianguis_limpio = df_tianguis.copy()

# Eliminar espacios adicionales en columnas de texto
for col in df_tianguis_limpio.select_dtypes(include="object").columns:
    df_tianguis_limpio[col] = df_tianguis_limpio[col].str.strip()

# Normalizar nombres de alcaldías
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(
        letra for letra in texto
        if not unicodedata.combining(letra)
    ).upper()

df_tianguis_limpio["alcaldia"] = (
    df_tianguis_limpio["alcaldia"].apply(normalizar)
)

# Normalizar días (incluyendo SÀBADO)
df_tianguis_limpio["dia"] = (
    df_tianguis_limpio["dia"].apply(normalizar)
)

# Normalizar formato de horarios
df_tianguis_limpio["horario"] = (
    df_tianguis_limpio["horario"]
    .str.replace(r"\b(\d):", r"0\1:", regex=True)
)

print("Registros:", len(df_tianguis_limpio))
print("Alcaldías diferentes:", df_tianguis_limpio["alcaldia"].nunique())
print("Duplicados exactos:", df_tianguis_limpio.duplicated().sum())

display(df_tianguis_limpio.head())

Registros: 1367
Alcaldías diferentes: 16
Duplicados exactos: 33


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4000\3053140774.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_tianguis_limpio.select_dtypes(include="object").columns:


,alcaldia,colonia,ubicacion,giro,dia,horario,latitud,longitud
0,ALVARO OBREGON,PILOTO (ADOLFO LOPEZ MATEOS) (AMPL),ADOLFO DE LA HUERTA ENTRE RUIZ CORTINEZ Y BENI...,NaN,LUNES,08:00 A 17:00 HORAS,19.375612,-99.221648
1,ALVARO OBREGON,GOLONDRINAS 1RA SECCION,AV. CHICAGO ENTRE URANO Y JUPITER,NaN,LUNES,08:00 A 17:00 HORAS,19.381123,-99.209121
2,ALVARO OBREGON,JOSE MARIA PINO SUAREZ,PONIENTE 85 Y AV. OBSERVATORIO,NaN,LUNES,08:00 A 17:00 HORAS,19.398888,-99.197936
3,ALVARO OBREGON,TLACUITLAPA,CALLE LAS TORRES Y ANTONIO MACHADO,NaN,LUNES,08:00 A 17:00 HORAS,19.355768,-99.240129
4,ALVARO OBREGON,MARIA G DE GARCIA RUIZ,AV. MEXICANOS ENT. BRASILEÃ‘OS Y URUGUAYOS,NaN,LUNES,08:00 A 17:00 HORAS,19.385972,-99.212968


In [13]:
# Verificar cantidad por alcaldía
print("ALCALDÍAS")
print(df_tianguis_limpio["alcaldia"].value_counts())

# Revisar los registros duplicados
print("\nEJEMPLOS DE DUPLICADOS")
duplicados = df_tianguis_limpio[
    df_tianguis_limpio.duplicated(keep=False)
]

display(duplicados.head(10))

ALCALDÍAS
alcaldia
IZTAPALAPA                360
GUSTAVO A. MADERO         206
COYOACAN                  137
TLALPAN                   135
ALVARO OBREGON            130
CUAUHTEMOC                 65
BENITO JUAREZ              59
IZTACALCO                  42
XOCHIMILCO                 42
TLAHUAC                    39
VENUSTIANO CARRANZA        37
AZCAPOTZALCO               35
MIGUEL HIDALGO             35
LA MAGDALENA CONTRERAS     23
CUAJIMALPA DE MORELOS      11
MILPA ALTA                 11
Name: count, dtype: int64

EJEMPLOS DE DUPLICADOS


,alcaldia,colonia,ubicacion,giro,dia,horario,latitud,longitud
294,CUAUHTEMOC,CUAUHTEMOC,JARDÍN DEL ARTE. “Bazar Jardín del Arte”,"PINTORES Y ESCULTORES, ARTICULOS PARA EL ARTE",DOMINGO,07:00 A 17:00 HORAS,19.433400,-99.161895
295,CUAUHTEMOC,CUAUHTEMOC,JARDÍN DEL ARTE. “Bazar Jardín del Arte”,"PINTORES Y ESCULTORES, ARTICULOS PARA EL ARTE",DOMINGO,07:00 A 17:00 HORAS,19.433400,-99.161895
299,CUAUHTEMOC,MORELOS III,"COMONFORT, DEL EJE 1 NORTE, A MATAMOROS “Tiang...",CHACHARAS,DOMINGO,07:00 A 17:00 HORAS,19.445567,-99.135021
300,CUAUHTEMOC,MORELOS III,"COMONFORT, DEL EJE 1 NORTE, A MATAMOROS “Tiang...",CHACHARAS,DOMINGO,07:00 A 17:00 HORAS,19.445567,-99.135021
307,GUSTAVO A. MADERO,C T M ARAGON (U),EXPLANADA LA HORMIGA ENTRE 602 Y ANDADOR 679,NaN,MARTES,07:00 A 17:00 HORAS,19.450991,-99.055901
308,GUSTAVO A. MADERO,C T M ARAGON (U),EXPLANADA LA HORMIGA ENTRE 602 Y ANDADOR 679,NaN,MARTES,07:00 A 17:00 HORAS,19.450991,-99.055901
327,GUSTAVO A. MADERO,SAN PEDRO EL CHICO,ANTONIO RUIZ GALINDO ENTRE NORTE 82-A Y NORTE ...,NaN,JUEVES,07:00 A 17:00 HORAS,19.471604,-99.098362
328,GUSTAVO A. MADERO,SAN PEDRO EL CHICO,ANTONIO RUIZ GALINDO ENTRE NORTE 82-A Y NORTE ...,NaN,JUEVES,07:00 A 17:00 HORAS,19.471604,-99.098362
331,GUSTAVO A. MADERO,SAN JUAN DE ARAGON 1A SECCION (U HAB) II,AV. 507 PLAZA 3,NaN,SABADO,07:00 A 17:00 HORAS,19.464014,-99.092910
338,GUSTAVO A. MADERO,CAMPESTRE ARAGON II,CAMINO PARQUE CENTRAL ENTRE CAMINO UNION Y CAM...,NaN,MIERCOLES,07:00 A 17:00 HORAS,19.486053,-99.077444


In [14]:
# Copiar los datos limpios
df_tianguis_final = df_tianguis_limpio.copy()

# Contar duplicados
duplicados = df_tianguis_final.duplicated().sum()

# Eliminar solamente filas idénticas
df_tianguis_final = df_tianguis_final.drop_duplicates().reset_index(drop=True)

print("Registros originales:", len(df_tianguis))
print("Duplicados eliminados:", duplicados)
print("Registros finales:", len(df_tianguis_final))

Registros originales: 1367
Duplicados eliminados: 33
Registros finales: 1334


In [15]:
# Revisar coordenadas faltantes
print("Coordenadas faltantes:")
print(df_tianguis_final[["latitud", "longitud"]].isna().sum())

# Revisar rangos geográficos
print("\nRangos de coordenadas:")
print(df_tianguis_final[["latitud", "longitud"]].describe())

# Coordenadas fuera de un rango aproximado de CDMX
fuera_rango = df_tianguis_final[
    ~df_tianguis_final["latitud"].between(19.0, 19.7) |
    ~df_tianguis_final["longitud"].between(-99.4, -98.8)
]

print("\nRegistros fuera del rango aproximado:", len(fuera_rango))
display(fuera_rango)

Coordenadas faltantes:
latitud     0
longitud    0
dtype: int64

Rangos de coordenadas:
           latitud     longitud
count  1334.000000  1334.000000
mean     19.345174   -99.129928
std       0.674268     0.137226
min       0.000000  -103.488744
25%      19.317534   -99.174200
50%      19.360915   -99.126430
75%      19.417785   -99.073644
max      19.569984   -98.952402

Registros fuera del rango aproximado: 2


,alcaldia,colonia,ubicacion,giro,dia,horario,latitud,longitud
1332,COYOACAN,CARMEN SERDÁN,BOULEVARD CARMEN SERDÁN ENTRE MAGDALENA COBOS ...,0,MIERCOLES,08:00 A 19:00 HORAS,4.420572,-99.102604
1333,COYOACAN,N/E,ACTUALMENTE NO SE INSTALA,0,VIERNES,08:00 A 19:00 HORAS,0.000000,-103.488744


In [16]:
display(
    fuera_rango[
        ["alcaldia", "colonia", "ubicacion",
         "dia", "horario", "latitud", "longitud"]
    ]
)

,alcaldia,colonia,ubicacion,dia,horario,latitud,longitud
1332,COYOACAN,CARMEN SERDÁN,BOULEVARD CARMEN SERDÁN ENTRE MAGDALENA COBOS ...,MIERCOLES,08:00 A 19:00 HORAS,4.420572,-99.102604
1333,COYOACAN,N/E,ACTUALMENTE NO SE INSTALA,VIERNES,08:00 A 19:00 HORAS,0.000000,-103.488744


In [17]:
import pandas as pd

# Copia de la base limpia
df_tianguis_final = df_tianguis_limpio.drop_duplicates().copy()

# Estado inicial
df_tianguis_final["estado"] = "ACTIVO_NO_VERIFICADO"

# Identificar coordenadas inválidas
coord_invalidas = (
    ~df_tianguis_final["latitud"].between(19.0, 19.7) |
    ~df_tianguis_final["longitud"].between(-99.4, -98.8)
)

df_tianguis_final.loc[coord_invalidas, "estado"] = "COORDENADAS_POR_VERIFICAR"

# Identificar registros que indican inactividad
inactivos = df_tianguis_final["ubicacion"].str.contains(
    "ACTUALMENTE NO SE INSTALA",
    case=False,
    na=False
)

df_tianguis_final.loc[inactivos, "estado"] = "NO_SE_INSTALA"

# Sustituir coordenadas inválidas por valores nulos
df_tianguis_final.loc[
    coord_invalidas, ["latitud", "longitud"]
] = float("nan")

print(df_tianguis_final["estado"].value_counts())
print("Registros totales:", len(df_tianguis_final))

estado
ACTIVO_NO_VERIFICADO         1332
COORDENADAS_POR_VERIFICAR       1
NO_SE_INSTALA                   1
Name: count, dtype: int64
Registros totales: 1334


In [18]:
from pathlib import Path

ruta_salida = Path("../Datos/procesados")
ruta_salida.mkdir(parents=True, exist_ok=True)

df_tianguis_final.to_csv(
    ruta_salida / "tianguis_cdmx_sedeco_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

df_tianguis_final.to_parquet(
    ruta_salida / "tianguis_cdmx_sedeco_2024.parquet",
    index=False
)

print("Archivos guardados correctamente")
print("Registros:", len(df_tianguis_final))

Archivos guardados correctamente
Registros: 1334


In [19]:
import pandas as pd

# Utilizamos la base filtrada de 86,931 registros
df_denue = pd.read_parquet(
    "../Datos/procesados/denue_4611_cdmx_2026.parquet"
)

print("REGISTROS:", len(df_denue))
print("COLUMNAS:", len(df_denue.columns))

print("\nDUPLICADOS")
print("Filas idénticas:", df_denue.duplicated().sum())
print("ID repetidos:", df_denue["id"].duplicated().sum())
print("CLEE repetidas:", df_denue["clee"].duplicated().sum())

print("\nVALORES FALTANTES")
print(df_denue.isna().sum().sort_values(ascending=False))

print("\nTIPOS DE UNIDAD ECONÓMICA")
print(df_denue["tipoUniEco"].value_counts(dropna=False))

print("\nCOORDENADAS")
print(df_denue[["latitud", "longitud"]].describe())

print("\nCOORDENADAS FUERA DE RANGO")
fuera_rango = df_denue[
    ~df_denue["latitud"].between(19.0, 19.7) |
    ~df_denue["longitud"].between(-99.4, -98.8)
]

print("Total:", len(fuera_rango))
display(fuera_rango.head(10))

REGISTROS: 86931
COLUMNAS: 42

DUPLICADOS
Filas idénticas: 0
ID repetidos: 0
CLEE repetidas: 0

VALORES FALTANTES
www           85551
edificio      85408
correoelec    82861
letra_int     80800
raz_social    79020
numero_int    75660
tipoCenCom    64460
num_local     64450
nom_CenCom    64439
edificio_e    64223
telefono      62075
letra_ext     44766
numero_ext    25825
nom_estab        25
nom_v_e_3         1
nom_v_e_1         1
nom_v_e_2         1
clee              0
tipo_v_e_1        0
nom_vial          0
tipo_vial         0
per_ocu           0
nombre_act        0
codigo_act        0
id                0
tipo_v_e_2        0
nomb_asent        0
tipo_asent        0
tipo_v_e_3        0
cod_postal        0
municipio         0
cve_ent           0
entidad           0
cve_mun           0
ageb              0
localidad         0
cve_loc           0
manzana           0
tipoUniEco        0
latitud           0
longitud          0
fecha_alta        0
dtype: int64

TIPOS DE UNIDAD ECONÓMICA
tipoUn

,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta


In [20]:
# Crear copia del DENUE filtrado
df_denue_final = df_denue.copy()

# Indicadores de información disponible
df_denue_final["tiene_web"] = df_denue_final["www"].notna()
df_denue_final["tiene_correo"] = df_denue_final["correoelec"].notna()
df_denue_final["tiene_telefono"] = df_denue_final["telefono"].notna()
df_denue_final["tiene_nombre"] = df_denue_final["nom_estab"].notna()

# Indicador de coordenadas dentro del rango preliminar
df_denue_final["coord_validas"] = (
    df_denue_final["latitud"].between(19.0, 19.7) &
    df_denue_final["longitud"].between(-99.4, -98.8)
)

print("Registros:", len(df_denue_final))
print("Columnas:", len(df_denue_final.columns))

display(
    df_denue_final[
        ["nom_estab", "codigo_act", "municipio",
         "tiene_web", "tiene_correo",
         "tiene_telefono", "coord_validas"]
    ].head()
)

Registros: 86931
Columnas: 47


,nom_estab,codigo_act,municipio,tiene_web,tiene_correo,tiene_telefono,coord_validas
0,NaN,461190,Miguel Hidalgo,True,True,False,True
1,NaN,461130,Iztapalapa,False,False,False,True
2,NaN,461130,Iztapalapa,False,False,False,True
3,NaN,461110,Xochimilco,False,False,True,True
4,NaN,461130,Iztapalapa,False,True,False,True


In [21]:
from pathlib import Path

ruta_salida = Path("../Datos/procesados")

df_denue_final.to_csv(
    ruta_salida / "denue_4611_cdmx_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

df_denue_final.to_parquet(
    ruta_salida / "denue_4611_cdmx_2026.parquet",
    index=False
)

print("Archivos actualizados correctamente")
print("Registros:", len(df_denue_final))
print("Columnas:", len(df_denue_final.columns))

Archivos actualizados correctamente
Registros: 86931
Columnas: 47
